# Importing relevant libraries

In [2]:
import pandas as pd
import numpy as np

# Importing the Car price prediciton dataset v3 and making a copy

In [4]:
df_init = pd.read_csv("Car details v3.csv")
df = df_init.copy()

In [5]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [6]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


Key Insights
- We have 13 attributes with selling_price as our target.
- There are clear signs of missing values given the less Non-Null count under seats, torque, etc
- The data types to many numeric attributes like engine, mileage, max_power, and torque are not integers or floats but rather objects due to units being included.

# Checking for duplicates and dropping them. (Data Cleaning)

In [9]:
df.duplicated().sum()

1202

In [10]:
(1202/8128)*100

14.788385826771652

- 1202 are exact duplicates accounting for 14.79% of the entire dataset. This is a significant number.
- They will be dropped to avoid model bias and potential data leakage.

In [12]:
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

In [13]:
df_no_duplicates.shape

(6926, 13)

## Changing text values into floating point values 

### Checking the fuel column and encoding it

In [16]:
df_no_duplicates["fuel"].unique()

array(['Diesel', 'Petrol', 'LPG', 'CNG'], dtype=object)

In [17]:
df_no_duplicates["fuel_encoded"] = pd.factorize(df_no_duplicates["fuel"])[0]

### Checking seller type

In [19]:
df_no_duplicates["seller_type"].unique()

array(['Individual', 'Dealer', 'Trustmark Dealer'], dtype=object)

In [20]:
df_no_duplicates["seller_type_encoded"] = pd.factorize(df_no_duplicates["seller_type"])[0]

### Checking transmission type

In [22]:
df_no_duplicates["transmission"].unique()

array(['Manual', 'Automatic'], dtype=object)

In [23]:
df_no_duplicates["transmission_encoded"] = pd.factorize(df_no_duplicates["transmission"])[0]

### Checking owner 

In [25]:
df_no_duplicates["owner"].unique()

array(['First Owner', 'Second Owner', 'Third Owner',
       'Fourth & Above Owner', 'Test Drive Car'], dtype=object)

In [26]:
df_no_duplicates["owner_encoded"] = pd.factorize(df_no_duplicates["owner"])[0]

### Checking Mileage 

In [28]:
# Extracting them into different columns for easy working using regex's
df_no_duplicates[["mileage_value", "mileage_unit"]] = df_no_duplicates["mileage"].str.extract(r'([\d.]+)\s*([a-zA-Z/]+)')

In [29]:
df_no_duplicates["mileage_value"] = pd.to_numeric(df_no_duplicates["mileage_value"], errors='coerce')

In [30]:
df_no_duplicates["mileage_unit"].unique()

array(['kmpl', 'km/kg', nan], dtype=object)

### Checking engine

In [32]:
df_no_duplicates[["engine_volume", "engine_v_units"]]=df_no_duplicates["engine"].str.extract(r'([\d.]+)\s*([a-zA-Z]+)')

In [33]:
df_no_duplicates["engine_volume"] = pd.to_numeric(df_no_duplicates["engine_volume"], errors='coerce')

In [34]:
df_no_duplicates["engine_v_units"].unique()

array(['CC', nan], dtype=object)

### Checking max_power 

In [36]:
df_no_duplicates[["max_power_value", "max_power_units"]]=df_no_duplicates["max_power"].str.extract(r'([\d.]+)\s*([a-zA-Z]+)')

In [37]:
df_no_duplicates["max_power_value"] = pd.to_numeric(df_no_duplicates["max_power_value"], errors='coerce')

In [38]:
df_no_duplicates["max_power_units"].unique()

array(['bhp', nan], dtype=object)

### Checking torque

In [40]:
df_no_duplicates["torque"].unique()

array(['190Nm@ 2000rpm', '250Nm@ 1500-2500rpm', '12.7@ 2,700(kgm@ rpm)',
       '22.4 kgm at 1750-2750rpm', '11.5@ 4,500(kgm@ rpm)',
       '113.75nm@ 4000rpm', '7.8@ 4,500(kgm@ rpm)', '59Nm@ 2500rpm',
       '170Nm@ 1800-2400rpm', '160Nm@ 2000rpm', '248Nm@ 2250rpm',
       '78Nm@ 4500rpm', nan, '84Nm@ 3500rpm', '115Nm@ 3500-3600rpm',
       '200Nm@ 1750rpm', '62Nm@ 3000rpm', '219.7Nm@ 1500-2750rpm',
       '114Nm@ 3500rpm', '115Nm@ 4000rpm', '69Nm@ 3500rpm',
       '172.5Nm@ 1750rpm', '6.1kgm@ 3000rpm', '114.7Nm@ 4000rpm',
       '60Nm@ 3500rpm', '90Nm@ 3500rpm', '151Nm@ 4850rpm',
       '104Nm@ 4000rpm', '320Nm@ 1700-2700rpm', '250Nm@ 1750-2500rpm',
       '145Nm@ 4600rpm', '146Nm@ 4800rpm', '343Nm@ 1400-3400rpm',
       '200Nm@ 1400-3400rpm', '200Nm@ 1250-4000rpm',
       '400Nm@ 2000-2500rpm', '138Nm@ 4400rpm', '360Nm@ 1200-3400rpm',
       '200Nm@ 1200-3600rpm', '380Nm@ 1750-2500rpm', '173Nm@ 4000rpm',
       '400Nm@ 1750-3000rpm', '400Nm@ 1400-2800rpm',
       '200Nm@ 1750-3000rp

In [41]:
torque_clean = df_no_duplicates['torque'].str.replace(',', '')

torque_val = pd.to_numeric(
    torque_clean.str.extract(r'^([\d.]+)')[0], 
    errors='coerce'
)

In [42]:
is_kgm = torque_clean.str.contains('kgm', case=False, na=False)

df_no_duplicates['torque_nm'] = np.where(is_kgm, torque_val * 9.80665, torque_val)

df_no_duplicates[['torque', 'torque_nm']].head(10)

,torque,torque_nm
0,190Nm@ 2000rpm,190.000000
1,250Nm@ 1500-2500rpm,250.000000
2,"12.7@ 2,700(kgm@ rpm)",124.544455
3,22.4 kgm at 1750-2750rpm,219.668960
4,"11.5@ 4,500(kgm@ rpm)",112.776475
5,113.75nm@ 4000rpm,113.750000
6,"7.8@ 4,500(kgm@ rpm)",76.491870
7,59Nm@ 2500rpm,59.000000
8,170Nm@ 1800-2400rpm,170.000000
9,160Nm@ 2000rpm,160.000000


In [43]:
df_no_duplicates.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,...,seller_type_encoded,transmission_encoded,owner_encoded,mileage_value,mileage_unit,engine_volume,engine_v_units,max_power_value,max_power_units,torque_nm
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,...,0,0,0,23.40,kmpl,1248.0,CC,74.00,bhp,190.000000
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,...,0,0,1,21.14,kmpl,1498.0,CC,103.52,bhp,250.000000
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,...,0,0,2,17.70,kmpl,1497.0,CC,78.00,bhp,124.544455
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,...,0,0,0,23.00,kmpl,1396.0,CC,90.00,bhp,219.668960
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,...,0,0,0,16.10,kmpl,1298.0,CC,88.20,bhp,112.776475


In [44]:
df_no_duplicates.columns

Index(['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner', 'mileage', 'engine', 'max_power', 'torque',
       'seats', 'fuel_encoded', 'seller_type_encoded', 'transmission_encoded',
       'owner_encoded', 'mileage_value', 'mileage_unit', 'engine_volume',
       'engine_v_units', 'max_power_value', 'max_power_units', 'torque_nm'],
      dtype='object')

# Feature Selection